# Neural IVP tutorial

Fill in the gaps.

The task: solve the first-order IVP
$$ \frac{dy}{dt} = \cos t, \qquad y(0) = 0, $$
with the neural IVP method, using only the ODE residual and the initial condition as losses. The analytic solution is
$$ y(t) = \sin t, $$
so you can check the trained network against it at the end.

## Recap of the method

1. A small MLP maps $\mathbf{t} \to \mathbf{y}(\mathbf{t})$ on a fixed time grid.
2. `torch.autograd.grad` computes the time derivative of $\mathbf{y}$.
3. The loss is the residual of the ODE and the initial condition (this is mathematically sufficient to solve the IVP for all time).
4. Adam minimises the loss by backpropagation.

In [1]:
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from IPython.display import display
from tqdm import tqdm

matplotlib.rcParams["font.family"] = "Times New Roman"
matplotlib.rcParams["font.size"] = 11
matplotlib.rcParams["mathtext.fontset"] = "stix"
matplotlib.rcParams["axes.linewidth"] = 0.8
matplotlib.rcParams["xtick.major.width"] = 0.8
matplotlib.rcParams["ytick.major.width"] = 0.8

DEVICE = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)
print(f"Using device: {DEVICE}")

Using device: mps


In [2]:
def f(t):
    """Right-hand side of dy/dt = f(t). You can optionally re-arrange the ODE such that f(t) = 0."""
    raise NotImplementedError("Implement f(t)")


t_check = torch.tensor([0.0], dtype=torch.float, device=DEVICE)
assert abs(f(t_check).item() - 1.0) < 1e-7, "f(0) should be cos(0) = 1"

NotImplementedError: Implement f(t)

In [ ]:
def analytic_solution(t):
    """Closed-form solution of the IVP."""
    raise NotImplementedError("Implement analytic_solution")


t_check = torch.tensor([0.0, torch.pi / 2.0], dtype=torch.float, device=DEVICE)
y_check = analytic_solution(t_check)
assert abs(y_check[0].item() - 0.0) < 1e-6, "y(0) must equal 0"
assert abs(y_check[1].item() - 1.0) < 1e-6, "y(pi/2) must equal 1"

In [ ]:
params = {
    "y0": 0.0,
    "T_MIN": 0.0,
    "T_MAX": 2.0 * torch.pi,
    "TIMESTEP": 0.05,
    "hidden": 64,
    "activation": "tanh",
    "epochs": 2000,
    "lr": 3e-3,
    "ODE_REGULARISER": 1.0,
    "IC_REGULARISER": 1.0,
}

dt = params["TIMESTEP"]
steps = int(round((params["T_MAX"] - params["T_MIN"]) / dt)) + 1
time_grid = torch.linspace(
    params["T_MIN"], params["T_MAX"], steps, dtype=torch.float, device=DEVICE
)
t = time_grid.unsqueeze(0).unsqueeze(2).requires_grad_(True)  # shape (1, T, 1)
y0 = params["y0"]

reference = analytic_solution(time_grid)
print(f"{steps} time points on t in [{params['T_MIN']}, {params['T_MAX']}], y(0) = {y0}")

In [ ]:
def gradient(y, x):
    """First derivative dy/dx via autograd."""
    raise NotImplementedError("Implement gradient(y, x)")

In [ ]:
def sechlu(x):
    """Sech-LU activation: sigmoid-gated linear."""
    return torch.sigmoid(2.0 * x) * x


class MLP(nn.Module):
    """3 hidden layer MLP."""

    def __init__(self, in_features, hidden, out_features, activation):
        super().__init__()
        self.L1 = ...
        raise NotImplementedError("Implement layers of the neural network")

    def _act(self, x):
        if self.activation == "tanh":
            return torch.tanh(x)
        if self.activation == "relu":
            return torch.relu(x)
        return sechlu(x)

    def forward(self, x):
        raise NotImplementedError("Implement the forward pass")


model = MLP(
    in_features=1,
    hidden=params["hidden"],
    out_features=1,
    activation=params["activation"],
).to(DEVICE)
optimiser = torch.optim.Adam(model.parameters(), lr=params["lr"])
loss_fn = torch.nn.MSELoss()

In [ ]:
def evaluate():
    prediction = model(t)
    # TODO: build the loss from the ODE residual and the initial condition, 
    # optionally add weighting to each term
    raise NotImplementedError("Implement evaluate()")
    return prediction, total_loss

In [ ]:
def make_axes():
    fig = plt.figure(figsize=(6, 4), dpi=150)
    fig.patch.set_facecolor("white")
    ax = fig.add_subplot(111)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(direction="in", which="both", left=True, bottom=True)
    ax.tick_params(labelbottom=True, labelleft=True)
    return fig, ax


def plot_trajectory(time_values, predicted, reference):
    fig, ax = make_axes()
    ax.plot(time_values, predicted, color="black", linewidth=2, label="Neural IVP")
    ax.plot(
        time_values,
        reference,
        color="lightgrey",
        linewidth=2.5,
        label="Analytic",
        zorder=-1,
    )
    ax.set_xlabel(r"$t$")
    ax.set_ylabel(r"$y(t)$")
    ax.legend(loc="upper right", frameon=False)
    return fig


def plot_loss_history(loss_history):
    fig, ax = make_axes()
    ax.semilogy(loss_history, linewidth=2, color="black")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    return fig

In [ ]:
loss_history = []

for epoch in tqdm(range(params["epochs"])):
    prediction, total_loss = evaluate()
    raise NotImplementedError("Implement the training step")

    if epoch % 500 == 0 or epoch == params["epochs"] - 1:
        time_values = time_grid.cpu().numpy()
        predicted_values = prediction[0, :, 0].detach().cpu().numpy()
        reference_values = reference.cpu().numpy()
        display(plot_trajectory(time_values, predicted_values, reference_values))
        display(plot_loss_history(loss_history))

In [ ]:
final_prediction, _ = evaluate()
time_values = time_grid.cpu().numpy()
predicted_values = final_prediction[0, :, 0].detach().cpu().numpy()
reference_values = reference.cpu().numpy()

max_error = None  # TODO: np.max(np.abs(predicted_values - reference_values))
assert max_error is not None, "Compute the max absolute error above"
print(f"Max |neural - analytic| over the interval: {max_error:.3e}")